# Pilot Notebook (T4) — Kurulum, EDA, Baseline, Küçük-Ölçek Eğitim

Bu notebook, Google Colab'da **T4 GPU** ile çalıştırılmak üzere tasarlanmıştır. Amacı, tüm
pipeline'ın (veri hazırlama -> EDA -> baseline değerlendirme -> LoRA eğitimi) küçük bir
veri alt kümesiyle **hatasız uçtan uca çalıştığını** doğrulamaktır (bir "duman testi").
Tam ölçekli eğitim için `01_full_training_a100.ipynb` kullanılır.

Çalıştırmadan önce Colab menüsünden: **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçili olmalıdır.

Hücreleri SIRAYLA çalıştırın.

## 1) Google Drive'ı bağla
Tüm kalıcı veri (ham/işlenmiş veri, checkpoint, log, değerlendirme sonuçları) Drive'da
tutulur; böylece Colab oturumu kapansa bile ilerleme kaybolmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Repoyu klonla ve bağımlılıkları kur

In [ ]:
import os

REPO_URL = "https://github.com/nidazeren/qwen2.5-vl.git"
REPO_DIR = "/content/qwen2.5-vl"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 3) Ortam değişkenleri: PILOT_MODE ve Drive kök klasörü
`QWEN_OCR_PILOT_MODE=1`, `configs/config.py` içindeki `PILOT_MODE` bayrağını `True` yapar
(küçük veri alt kümesi, 1 epoch, 4-bit yükleme, küçük batch — T4'e uygun ayarlar).

In [ ]:
import os, sys

os.environ["QWEN_OCR_PILOT_MODE"] = "1"
os.environ["QWEN_OCR_DRIVE_ROOT"] = "/content/drive/MyDrive/qwen25vl_turkish_ocr"
sys.path.insert(0, REPO_DIR)

from configs import config
config.ensure_directories()
print("PILOT_MODE:", config.PILOT_MODE)
print("DRIVE_ROOT:", config.DRIVE_ROOT)
print("Hedef kova boyutlari (pilot):", config.compute_bucket_target_sizes())

## 4) Kaggle API kimlik bilgisi (TS-TR veri seti için gerekli)
Kaggle hesabınızdan **Settings > API > Create New Token** ile bir `kaggle.json` indirin.
Bu hücre, dosyayı bir kere Drive'a kaydedip sonraki çalıştırmalarda tekrar yüklemenizi
gerektirmez.

In [ ]:
import os, shutil, stat

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
target = os.path.join(kaggle_dir, "kaggle.json")
drive_copy = str(config.DRIVE_ROOT / "kaggle.json")

if os.path.exists(drive_copy):
    shutil.copy(drive_copy, target)
    print("kaggle.json Drive'dan kopyalandi.")
elif os.path.exists(target):
    print("kaggle.json zaten mevcut.")
else:
    from google.colab import files
    print("Lutfen kaggle.json dosyanizi secin:")
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, target)
    shutil.copy(target, drive_copy)  # sonraki oturumlar icin Drive'a da kopyala

os.chmod(target, stat.S_IRUSR | stat.S_IWUSR)
print("Kaggle kimlik bilgisi hazir:", target)

## 5) (Opsiyonel) SMHD el yazısı verisi
`configs/config.py` içinde `USE_SMHD = True` ise, `hiqmatNisa/SMHD` GitHub reposundaki
izin formunu doldurup indirdiğiniz veriyi şu klasöre yerleştirmelisiniz:
`config.SMHD_LOCAL_DIR` (varsayılan: `.../qwen25vl_turkish_ocr/manual_datasets/SMHD`).
Varsayılan `USE_SMHD=False` iken bu adımı atlayabilirsiniz; kod otomatik olarak
el yazısı payının tamamını `emredeveloper/turkish-ocr` kaynağına kaydırır.

In [ ]:
print("USE_SMHD =", config.USE_SMHD)
print("Beklenen SMHD klasoru:", config.SMHD_LOCAL_DIR)

## 6) Veri hazırlama (pilot alt küme)
Tüm ham kaynakları indirir/normalize eder; `PILOT_MODE=True` olduğundan her kaynaktan
yalnızca `config.PILOT_SAMPLES_PER_SOURCE` kadar örnek kullanılır (hızlı çalışsın diye).

In [ ]:
!python data/prepare_datasets.py

## 7) EDA: Tokenizer analizi ve görsel token sayısı
Tokenizer analizi, `ENABLE_EMBED_LORA` için bir öneri üretir (raporu okuyup kararı siz
verirsiniz). Görsel token EDA'sı, `MIN_PIXELS`/`MAX_PIXELS` seçiminize yardımcı olur.

In [ ]:
!python analysis/tokenizer_analysis.py

In [ ]:
!python analysis/vision_token_eda.py

## 8) Self-distillation replay verisi üretimi
Taban model (henüz LoRA uygulanmadan), OmniDocBench görselleri üzerinde çalıştırılıp
kendi çıktıları "replay" hedef metni olarak kaydedilir. Bu adım GPU kullanır ve pilot
modda bile birkaç dakika sürebilir.

In [ ]:
!python data/replay_generation.py

## 9) Eğitim/Validation/Test A/Test B setlerini oluştur

In [ ]:
!python data/build_chat_dataset.py

## 10) Baseline değerlendirme
LoRA henüz uygulanmadığı için burada taban modelin Test A/B skorları ölçülür ve
`eval_outputs/baseline.json` olarak kaydedilir. (training/train_sft.py, bu dosya yoksa
kendisi de otomatik üretir; ama pilot akışında ayrıca burada da görmek isteyebilirsiniz.)

In [ ]:
!python evaluation/evaluate.py --tag baseline

## 11) Pilot LoRA eğitimi (T4, 4-bit + gradient checkpointing)
`PILOT_MODE=True` olduğundan: küçük veri, 1 epoch, düşük batch + yüksek gradient
accumulation, 4-bit yükleme. Bu adımın hatasız tamamlanması, A100'de tam eğitime
geçmeden önceki nihai doğrulamadır.

In [ ]:
!python training/train_sft.py

## 12) Sonuçları incele
`config.EVAL_OUTPUT_DIR` altındaki `baseline.json`, `epoch_1.json` (ve varsa
`regression_report.json`) dosyalarına bakın. Pilot başarılıysa (hata vermeden
tamamlandıysa) `01_full_training_a100.ipynb`'ye geçebilirsiniz.

In [ ]:
import json

for name in ["baseline.json", "epoch_1.json", "regression_report.json"]:
    path = config.EVAL_OUTPUT_DIR / name
    if path.exists():
        print(f"--- {name} ---")
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    else:
        print(f"--- {name} yok (bu normal olabilir) ---")